In [ ]:
EXP_NAME = "prev40_dim4_cont_SPS_norm"
dataset_path = "synth/prev40_dim4_cont/train"
feature_map = "sps/toy_features_4"

# ORIGINAL_SCM = ""
ORIGINAL_SCM = "ORIGINAL SCM"

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
  from google.colab import userdata
  from google.colab import drive
  drive.mount('/content/drive')
  PROJECT_ROOT = userdata.get('PROJECT_ROOT')
else:
  PROJECT_ROOT = '../..'
  if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
from src.config import Config
from notebooks.analysis_utils import min_max_scale, get_pareto_frontier, plot_pareto_frontier, count_feature_appearance

sns.set_style('whitegrid')
sns.set_context('paper', font_scale=1.2)

sps_audit = pd.read_csv(f'{PROJECT_ROOT}/{Config.RESULTS_DIR}/{EXP_NAME}/sps_audit.csv')
sbs_audit_baseline = pd.read_csv(f'{PROJECT_ROOT}/{Config.RESULTS_DIR}/{EXP_NAME}/sps_audit_baseline.csv')
dataset = pd.read_csv(f'{PROJECT_ROOT}/{Config.DATA_DIR}/{dataset_path}.csv')

with open(f"{PROJECT_ROOT}/configs/{feature_map}.json", 'r') as f:
  features = json.load(f)

# Analysis setup

In [ ]:
full_audit = pd.concat([sps_audit, sbs_audit_baseline], ignore_index=True, axis=0)
x_desc_configs = full_audit[full_audit['bucket'] == 'x_desc'].groupby(['iteration'])['feature'].apply(set).to_dict()

iteration_per_feature = full_audit[full_audit['bucket'] == 'x_desc'].groupby('feature')[['iteration']].count()
# iteration_per_feature

In [ ]:
subgroups = dataset[features["sens"][0]["name"]].unique().astype(int)

new_metrics = {}

# UTILITY VALUE
new_metrics['utility_value'] = full_audit['u_global_mean_auprc'] - full_audit['abla_global_mean_auprc'] # MAXIMISE

# UTILITY COST
new_metrics['utility_cost'] = full_audit['x_global_mean_auprc'] - full_audit['u_global_mean_auprc'] # MINIMISE

new_metrics['s_utility_cost'] = min_max_scale(new_metrics['utility_cost'].values) # Low cost -> 0, High cost -> 1

# FAIRNESS GAIN: Equalised Odds Ratio Delta
# new_metrics['fairness_gain'] = full_audit['u_global_mean_eor'] - full_audit['x_global_mean_eor']
# FAIRNESS GAIN: Equalised Odds Diff Delta
new_metrics['fairness_gain'] = full_audit['x_global_mean_eod'] - full_audit['u_global_mean_eod'] # MAXIMISE

new_metrics['s_fairness_gain'] = min_max_scale(new_metrics['fairness_gain'].values) #High fairness gain -> 1, Low fairness gain -> 0

# RAWLS FAIRNESS GAIN: 
u_min = pd.Series(1.0, index=full_audit.index)
x_min = pd.Series(1.0, index=full_audit.index)
for g in subgroups:
  u_min = np.minimum(full_audit[f"u_{g}_mean_f1"], u_min)
  x_min = np.minimum(full_audit[f"x_{g}_mean_f1"], x_min)

new_metrics['rawls_fairness_gain'] = u_min - x_min # MAXIMISE
new_metrics['s_rawls_fairness_gain'] = min_max_scale(new_metrics['rawls_fairness_gain'].values) #High fairness gain -> 1, Low fairness gain -> 0

# Statistical independence from S
# S probe AUPRC delta
new_metrics['sens_info_reduction'] = (full_audit['mean_x_s_bal_acc'] - 0.5).abs() - (full_audit['mean_u_s_bal_acc'] - 0.5).abs() # MAXIMISE
new_metrics['s_sens_info_reduction'] = min_max_scale(new_metrics['sens_info_reduction'].values) # High reduction -> 1, Low reduction -> 0

# CF harm reduction
cf_harm_max = pd.Series(0.0, index=full_audit.index)
for g in subgroups:
  cf_harm_max = np.maximum(full_audit[f'x_{g}_bal_harm_mean'], cf_harm_max)
new_metrics['max_cf_harm_reduction'] = cf_harm_max # MAXIMISE
new_metrics['s_max_cf_harm_reduction'] = min_max_scale(new_metrics['max_cf_harm_reduction'].values) # High reduction -> 1, Low reduction -> 0

new_cols_df = pd.DataFrame(new_metrics, index=full_audit.index)
full_audit = pd.concat([full_audit, new_cols_df], axis=1)


# Config trade-off analysis

In [ ]:
tradeoff_audit = full_audit.groupby('iteration')[['utility_cost', 'fairness_gain', 's_utility_cost', 's_fairness_gain', 'utility_value', 's_rawls_fairness_gain', 'rawls_fairness_gain', 's_sens_info_reduction', 'max_cf_harm_reduction', 's_max_cf_harm_reduction']].median().round(4)

def find_config(idx):
  return x_desc_configs.get(idx, "empty Xdesc")

tradeoff_audit['Xdesc config'] = tradeoff_audit.index.map(find_config)

In [ ]:
def calc_feu(row, fairness_metric, utility_metric):
  return round(row[utility_metric] / row[fairness_metric], 3) if row[fairness_metric] else np.nan

In [ ]:
w = {'utility_cost': 1, 'fairness_gain': 1}

def calc_dist_to_ideal(row, fairness_metric, utility_metric):
  """
  Calculate weighted squared distance to the absolute ideal point (utility cost: 0, fairness gain: 1)
  """
  d2 = ( 
    w['utility_cost'] * (row[utility_metric])**2 +
    w['fairness_gain'] * (row[fairness_metric] - 1)**2
  )
  return round(np.sqrt(d2), 4)

## Fairness as Equalized Odds

In [ ]:
utility_vs_eod_fairness = tradeoff_audit.copy()

In [ ]:
# FEU
utility_vs_eod_fairness['FEU'] = utility_vs_eod_fairness.apply(lambda x: calc_feu(x, 's_fairness_gain', 's_utility_cost'), axis=1)

# Distance to ideal fairness gain and utility cost
utility_vs_eod_fairness['dist_to_ideal'] = utility_vs_eod_fairness.apply(lambda x: calc_dist_to_ideal(x, 's_fairness_gain', 's_utility_cost'), axis=1)

In [ ]:
eod_pareto_frontier_df = get_pareto_frontier(
  utility_vs_eod_fairness,
  s_metrics=['s_fairness_gain', 's_utility_cost'],
  minimise=[False, True],
  condition=lambda x: x['utility_value'] > 0
)

print(eod_pareto_frontier_df.to_markdown())

In [ ]:
best_feu = eod_pareto_frontier_df[eod_pareto_frontier_df['FEU'] == eod_pareto_frontier_df['FEU'].min()]
print(f"Config with best FEU on the Frontier: {[x_desc_configs.get(idx, "empty") for idx in best_feu.index.values]}")

In [ ]:
best_dist = eod_pareto_frontier_df[eod_pareto_frontier_df['dist_to_ideal'] == eod_pareto_frontier_df['dist_to_ideal'].min()]
print(f"Config closest to ideal trade-off on the Frontier: {[x_desc_configs.get(idx, "empty") for idx in best_dist.index.values]}")

In [ ]:
plot_pareto_frontier(
  eod_pareto_frontier_df, 
  utility_vs_eod_fairness,
  s_metrics=['s_fairness_gain', 's_utility_cost'],
  labels=['EOD Fairness Gain (Maximise)', 'Utility Cost (Minimise)'],
  original_scm=ORIGINAL_SCM
)

plt.show()

### Original SCM Ranking

In [ ]:
all_configs = utility_vs_eod_fairness.reset_index(names="iteration")

In [ ]:
all_configs_by_dist_to_ideal = all_configs.sort_values(by="dist_to_ideal").reset_index(drop=True)

if ORIGINAL_SCM and ORIGINAL_SCM != "":
  print(f"ORIGINAL SCM Config Rank by distance ot ideal point (out of {len(all_configs_by_dist_to_ideal)})")
  print("="*50)
  print(all_configs_by_dist_to_ideal[all_configs_by_dist_to_ideal['iteration'] == ORIGINAL_SCM].to_markdown())

In [ ]:
print("Full ranking by distance to ideal point:")
print("="*50)
print(all_configs_by_dist_to_ideal[['iteration', 'Xdesc config']].to_markdown())

In [ ]:
all_configs_by_feu = all_configs.sort_values(by="FEU").reset_index(drop=True)

if ORIGINAL_SCM and ORIGINAL_SCM != "":
  print(f"ORIGINAL SCM Config Rank by FEU (out of {len(all_configs_by_feu)})")
  print("="*50)
  print(all_configs_by_feu[all_configs_by_feu['iteration'] == ORIGINAL_SCM].to_markdown())


In [ ]:
print("Full ranking by FEU:")
print("="*50)
print(all_configs_by_feu['iteration'].to_markdown())

### Features on the Frontier

In [ ]:
print(f"Frontier size: {len(eod_pareto_frontier_df)}")
eod_pareto_frontier_df['Xdesc config']

for f in features['X']:
  count = count_feature_appearance(eod_pareto_frontier_df, f['name'])
  print(f"{f['name']}: {count}")

---

## Fairness as Rawls Minimum F1 score uplift

In [ ]:
utility_vs_rawls_fairness = tradeoff_audit.copy()

In [ ]:
# FEU
utility_vs_rawls_fairness['FEU'] = utility_vs_rawls_fairness.apply(lambda x: calc_feu(x, 's_rawls_fairness_gain', 's_utility_cost'), axis=1)

# Distance to ideal fairness gain and utility cost
utility_vs_rawls_fairness['dist_to_ideal'] = utility_vs_rawls_fairness.apply(lambda x: calc_dist_to_ideal(x, 's_rawls_fairness_gain', 's_utility_cost'), axis=1)

In [ ]:
rawls_pareto_frontier_df = get_pareto_frontier(
  utility_vs_rawls_fairness,
  s_metrics=['s_rawls_fairness_gain', 's_utility_cost'],
  minimise=[False, True],
  condition=lambda x: x['utility_value'] > 0
)

print(rawls_pareto_frontier_df.to_markdown())

In [ ]:
best_feu = rawls_pareto_frontier_df[rawls_pareto_frontier_df['FEU'] == rawls_pareto_frontier_df['FEU'].min()]
print(f"Config with best FEU: {[x_desc_configs.get(idx, "empty") for idx in best_feu.index.values]}")

In [ ]:
best_dist = rawls_pareto_frontier_df[rawls_pareto_frontier_df['dist_to_ideal'] == rawls_pareto_frontier_df['dist_to_ideal'].min()]
print(f"Config closest to ideal trade-off: {[x_desc_configs.get(idx, "empty") for idx in best_dist.index.values]}")

In [ ]:
plot_pareto_frontier(
  rawls_pareto_frontier_df, 
  utility_vs_rawls_fairness,
  s_metrics=['s_rawls_fairness_gain', 's_utility_cost'],
  labels=['Rawls Fairness Gain (Maximise)', 'Utility Cost (Minimise)'],
  original_scm=ORIGINAL_SCM
)

plt.show()

### Original SCM Ranking

In [ ]:
all_configs = utility_vs_rawls_fairness.reset_index(names="iteration")

In [ ]:
if ORIGINAL_SCM and ORIGINAL_SCM != "":
  all_configs_by_dist_to_ideal = all_configs.sort_values(by="dist_to_ideal").reset_index(drop=True)

  print(f"ORIGINAL SCM Config Rank by distance ot ideal point (out of {len(all_configs_by_dist_to_ideal)})")
  print("="*50)
  print(all_configs_by_dist_to_ideal[all_configs_by_dist_to_ideal['iteration'] == ORIGINAL_SCM].to_markdown())

In [ ]:
print("Full ranking by distance to ideal point:")
print("="*50)
print(all_configs_by_dist_to_ideal['iteration'].to_markdown())

In [ ]:
if ORIGINAL_SCM and ORIGINAL_SCM != "":
  all_configs_by_feu = all_configs.sort_values(by="FEU").reset_index(drop=True)

  print(f"ORIGINAL SCM Config Rank by FEU (out of {len(all_configs_by_feu)})")
  print("="*50)
  print(all_configs_by_feu[all_configs_by_feu['iteration'] == ORIGINAL_SCM].to_markdown())


In [ ]:
print("Full ranking by FEU:")
print("="*50)
print(all_configs_by_feu['iteration'].to_markdown())

### Features on the Frontier

In [ ]:
print(f"Frontier size: {len(rawls_pareto_frontier_df)}")
rawls_pareto_frontier_df['Xdesc config']

for f in features['X']:
  count = count_feature_appearance(rawls_pareto_frontier_df, f['name'])
  print(f"{f['name']}: {count}")

---

## Utility vs S-info reduction

In [ ]:
utility_vs_s_info = tradeoff_audit.copy()

In [ ]:
# FEU
utility_vs_s_info['FEU'] = utility_vs_s_info.apply(lambda x: calc_feu(x, 's_sens_info_reduction', 's_utility_cost'), axis=1)

# Distance to ideal fairness gain and utility cost
utility_vs_s_info['dist_to_ideal'] = utility_vs_s_info.apply(lambda x: calc_dist_to_ideal(x, 's_sens_info_reduction', 's_utility_cost'), axis=1)

In [ ]:
s_info_pareto_frontier_df = get_pareto_frontier(
  utility_vs_s_info,
  s_metrics=['s_sens_info_reduction', 's_utility_cost'],
  minimise=[False, True],
  condition=lambda x: x['utility_value'] > 0
)

print(s_info_pareto_frontier_df.to_markdown())

In [ ]:
best_feu = s_info_pareto_frontier_df[s_info_pareto_frontier_df['FEU'] == s_info_pareto_frontier_df['FEU'].min()]
print(f"Config with best FEU on the Frontier: {[x_desc_configs.get(idx, "empty") for idx in best_feu.index.values]}")

In [ ]:
best_dist = s_info_pareto_frontier_df[s_info_pareto_frontier_df['dist_to_ideal'] == s_info_pareto_frontier_df['dist_to_ideal'].min()]
print(f"Config closest to ideal trade-off on the Frontier: {[x_desc_configs.get(idx, "empty") for idx in best_dist.index.values]}")

In [ ]:
plot_pareto_frontier(
  s_info_pareto_frontier_df, 
  utility_vs_s_info,
  s_metrics=['s_sens_info_reduction', 's_utility_cost'],
  labels=['Sens. Attribute Info Reduction (Maximise)', 'Utility Cost (Minimise)'],
  original_scm=ORIGINAL_SCM
)

plt.show()

### Original SCM ranking

In [ ]:
all_configs = utility_vs_s_info.reset_index(names="iteration")

In [ ]:
if ORIGINAL_SCM and ORIGINAL_SCM != "":
  all_configs_by_dist_to_ideal = all_configs.sort_values(by="dist_to_ideal").reset_index(drop=True)

  print(f"ORIGINAL SCM Config Rank by distance ot ideal point (out of {len(all_configs_by_dist_to_ideal)})")
  print("="*50)
  print(all_configs_by_dist_to_ideal[all_configs_by_dist_to_ideal['iteration'] == ORIGINAL_SCM].to_markdown())

In [ ]:
print("Full ranking by distance to ideal point:")
print("="*50)
print(all_configs_by_dist_to_ideal['iteration'].to_markdown())

In [ ]:
if ORIGINAL_SCM and ORIGINAL_SCM != "":
  all_configs_by_feu = all_configs.sort_values(by="FEU").reset_index(drop=True)

  print(f"ORIGINAL SCM Config Rank by FEU (out of {len(all_configs_by_feu)})")
  print("="*50)
  print(all_configs_by_feu[all_configs_by_feu['iteration'] == ORIGINAL_SCM].to_markdown())


In [ ]:
print("Full ranking by FEU:")
print("="*50)
print(all_configs_by_feu['iteration'].to_markdown())

### Features on the Frontier

In [ ]:
print(f"Frontier size: {len(s_info_pareto_frontier_df)}")
s_info_pareto_frontier_df['Xdesc config']

for f in features['X']:
  count = count_feature_appearance(s_info_pareto_frontier_df, f['name'])
  print(f"{f['name']}: {count}")

---

## Fairness as CF harm reduction

In [ ]:
utility_vs_cf_harm_reduc = tradeoff_audit.copy()

In [ ]:
# FEU
utility_vs_cf_harm_reduc['FEU'] = utility_vs_cf_harm_reduc.apply(lambda x: calc_feu(x, 's_max_cf_harm_reduction', 's_utility_cost'), axis=1)

# Distance to ideal fairness gain and utility cost
utility_vs_cf_harm_reduc['dist_to_ideal'] = utility_vs_cf_harm_reduc.apply(lambda x: calc_dist_to_ideal(x, 's_max_cf_harm_reduction', 's_utility_cost'), axis=1)

In [ ]:
cf_harm_pareto_frontier_df = get_pareto_frontier(
  utility_vs_cf_harm_reduc,
  s_metrics=['s_max_cf_harm_reduction', 's_utility_cost'],
  minimise=[False, True],
  condition=lambda x: x['utility_value'] > 0
)

print(cf_harm_pareto_frontier_df.to_markdown())

In [ ]:
best_feu = cf_harm_pareto_frontier_df[cf_harm_pareto_frontier_df['FEU'] == cf_harm_pareto_frontier_df['FEU'].min()]
print(f"Config with best FEU on the Frontier: {[x_desc_configs.get(idx, "empty") for idx in best_feu.index.values]}")

best_dist = cf_harm_pareto_frontier_df[cf_harm_pareto_frontier_df['dist_to_ideal'] == cf_harm_pareto_frontier_df['dist_to_ideal'].min()]
print(f"Config closest to ideal trade-off on the Frontier: {[x_desc_configs.get(idx, "empty") for idx in best_dist.index.values]}")

In [ ]:
plot_pareto_frontier(
  cf_harm_pareto_frontier_df, 
  utility_vs_cf_harm_reduc,
  s_metrics=['s_max_cf_harm_reduction', 's_utility_cost'],
  labels=['CF harm reduction (Maximise)', 'Utility Cost (Minimise)'],
  original_scm=ORIGINAL_SCM
)

plt.show()

### Original SCM Ranking

In [ ]:
all_configs = utility_vs_cf_harm_reduc.reset_index(names="iteration")

In [ ]:
all_configs_by_dist_to_ideal = all_configs.sort_values(by="dist_to_ideal").reset_index(drop=True)

if ORIGINAL_SCM and ORIGINAL_SCM != "":
  print(f"ORIGINAL SCM Config Rank by distance ot ideal point (out of {len(all_configs_by_dist_to_ideal)})")
  print("="*50)
  print(all_configs_by_dist_to_ideal[all_configs_by_dist_to_ideal['iteration'] == ORIGINAL_SCM].to_markdown())

In [ ]:
print("Full ranking by distance to ideal point:")
print("="*50)
print(all_configs_by_dist_to_ideal[['iteration', 'Xdesc config']].to_markdown())

In [ ]:
all_configs_by_feu = all_configs.sort_values(by="FEU").reset_index(drop=True)

if ORIGINAL_SCM and ORIGINAL_SCM != "":
  print(f"ORIGINAL SCM Config Rank by FEU (out of {len(all_configs_by_feu)})")
  print("="*50)
  print(all_configs_by_feu[all_configs_by_feu['iteration'] == ORIGINAL_SCM].to_markdown())

In [ ]:
print("Full ranking by FEU:")
print("="*50)
print(all_configs_by_feu['iteration'].to_markdown())

### Features on the Frontier

In [ ]:
print(f"Frontier size: {len(cf_harm_pareto_frontier_df)}")
cf_harm_pareto_frontier_df['Xdesc config']

for f in features['X']:
  count = count_feature_appearance(cf_harm_pareto_frontier_df, f['name'])
  print(f"{f['name']}: {count}")

---

# Deep dive

## Equalised Odds Difference

In [ ]:
eod_deep_dive = full_audit.groupby(['iteration', 'run_id'])[[ 'x_global_mean_eod', 'u_global_mean_eod', 'fairness_gain']].first().reset_index(level=0).reset_index(drop=True)

In [ ]:
unique_iterations = eod_deep_dive['iteration'].unique()

color_map = {
  i: "orange" if i == ORIGINAL_SCM else "#999999"
  for i in unique_iterations
}
colors = [color_map[i] for i in unique_iterations]

plt.figure(figsize=(10, 6))
ax = pd.plotting.parallel_coordinates(
    eod_deep_dive[['x_global_mean_eod', 'u_global_mean_eod', 'iteration']], class_column="iteration", color=colors, alpha=0.7, lw=2
)

handles, labels = ax.get_legend_handles_labels()
new_handles = [None, None]
new_labels = ["",""]
has_gray_legend = False

for handle, label in zip(handles, labels):
    if label == ORIGINAL_SCM:
        new_handles[0] = handle
        new_labels[0] = label
    elif not has_gray_legend:
        new_handles[1] = handle
        new_labels[1] = "Other Iterations"
        has_gray_legend = True

ax.legend(new_handles, new_labels, loc="upper right")

plt.show()
